## Importing Required Libraries

In this block, we import all the necessary libraries for the project.

- `os` is used for handling file paths and directories.
- `pandas` and `numpy` are used for data manipulation and numerical operations.
- `matplotlib` is used for visualization.
- `tensorflow` and `keras` are used to build and train the deep learning model.
- Keras layers like Conv2D, MaxPooling2D, Flatten, Dense, and Dropout are used to construct the CNN architecture.
- `ImageDataGenerator` is used to efficiently load and preprocess image data.
- `train_test_split` is used to divide the dataset into training and testing sets.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense,Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split


**Load & Prepare Data**

## Loading and Organizing the Dataset

In this block, we define the dataset directory and categories.

- The dataset contains two classes:
  - `yes` → images with tumors
  - `no` → images without tumors
- We iterate through both folders and collect:
  - Image file paths
  - Corresponding labels
- These are stored in separate lists for further processing.

In [ ]:
data_dir = '/kaggle/input/brain-tumor-detection'
groups = ['yes', 'no']
file_paths = []
labels = []

for group in groups:
    fold_path = os.path.join(data_dir, group) 
    files = os.listdir(fold_path)
    for file in files:
        file_path = os.path.join(fold_path, file)
        file_paths.append(file_path)
        labels.append(group)


## Creating a DataFrame

Here, we convert the collected file paths and labels into a structured DataFrame.

- This makes it easier to manage and manipulate the dataset.
- We also:
  - Display random samples from the dataset
  - Check the distribution of classes (tumor vs no tumor)

In [ ]:
df=pd.DataFrame({'file_paths':file_paths,
                'labels':labels})

print(df.sample(10))

print(df['labels'].value_counts())


## Train/Test Split

In [ ]:
train_df,test_df=train_test_split(df,test_size=0.2,random_state=42,stratify=df['labels'])

## Image Data Preprocessing

We use ImageDataGenerator to prepare the image data for training.

- Images are resized to a fixed size (224x224)
- Converted to grayscale
- Loaded in batches to improve performance
- Labels are assigned automatically
- This approach avoids loading the entire dataset into memory at once

In [ ]:
gen=ImageDataGenerator()
train_gen=gen.flow_from_dataframe(train_df, x_col='file_paths', y_col='labels', 
                                    target_size=(224, 224), color_mode='grayscale', 
                                    class_mode='binary', batch_size=16)

test_gen=gen.flow_from_dataframe(test_df, x_col='file_paths', y_col='labels', 
                                    target_size=(224, 224), color_mode='grayscale', 
                                    class_mode='binary', batch_size=16)


## Building the Convolutional Neural Network (CNN)

In this block, we define the CNN architecture.

- Convolutional layers extract features from images
- MaxPooling layers reduce spatial dimensions
- Dropout layers help prevent overfitting
- Flatten layer converts 2D data into 1D
- Dense layers perform classification
- The final layer uses sigmoid activation for binary classification (tumor / no tumor)

In [ ]:
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 1)),
    MaxPooling2D(2, 2),
    Dropout(0.25),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Dropout(0.25),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),

    Flatten(),

    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),

    Dense(1, activation='sigmoid')
])




## Compiling the Model

Here, we configure the learning process.

- `optimizer='adam'` is used for efficient training
- `loss='binary_crossentropy'` is used for binary classification
- `accuracy` is used as the evaluation metric

In [ ]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

## Training the Model

In this step, we train the CNN model.

- The model learns patterns from training images
- Training runs for multiple epochs
- Validation is performed using test data
- Training history is stored for later analysis

In [ ]:
traning = model.fit(train_gen, epochs=35, validation_data=test_gen)

## Evaluating Model Performance

We evaluate the trained model on both training and testing datasets.

- This helps measure:
  - Accuracy
  - Loss
- It also helps identify overfitting or underfitting

In [ ]:
model.evaluate(train_gen)

In [ ]:
model.evaluate(test_gen)

## Making Predictions on New Images

We define a function to predict whether a given image contains a tumor.

- The image is loaded and resized
- Converted into an array format
- Passed through the trained model
- Prediction is made based on probability
- The result is displayed along with the image

In [ ]:
plt.figure(figsize=(7,5))
plt.plot(traning.history['accuracy'], label='Train Accuracy')
plt.plot(traning.history['val_accuracy'], label='Validation Accuracy')
plt.legend()
plt.title('Model Accuracy')
plt.show()

plt.figure(figsize=(7,5))
plt.plot(traning.history['loss'], label='Train Loss')
plt.plot(traning.history['val_loss'], label='Validation Loss')
plt.legend()
plt.title('Model Loss')

plt.show()




## Testing with Sample Images

In this block, we randomly select images from both classes.

- Predictions are made on these images
- Results are displayed to visually verify model performance

In [ ]:
import random
from tensorflow.keras.preprocessing import image

def show_prediction(folder, filename):
    img_path = os.path.join(data_dir, folder, filename)
    img = image.load_img(img_path, target_size=(224,224), color_mode='grayscale')
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0) 
    prediction = model.predict(img_array)

    if prediction[0][0] > 0.5:
        result = "Tumor"
        color = "red"
    else:
        result = "No Tumor"
        color = "green"

    plt.imshow(np.array(img).squeeze(), cmap='gray')
    plt.title(f"Actual: {folder.upper()}\nPredicted: {result}", color=color)
    plt.axis('off')


yes_images = random.sample(os.listdir(os.path.join(data_dir, 'yes')), 3)
no_images  = random.sample(os.listdir(os.path.join(data_dir, 'no')), 3)


plt.figure(figsize=(12, 8))

for i, img_name in enumerate(yes_images):
    plt.subplot(2, 3, i + 1)
    show_prediction('yes', img_name)

for i, img_name in enumerate(no_images):
    plt.subplot(2, 3, i + 4)
    show_prediction('no', img_name)

plt.tight_layout()
plt.show()

## Saving the Trained Model

Finally, we save the trained model to a file.

- This allows us to reuse the model later
- No need to retrain the model every time
- The model can be deployed in applications

In [ ]:
model.save("brain_tumor_model.h5")